# AI Programming — Lecture 19
## Lab 4-4a: Decoder-only Transformer for ETTh1 Forecasting
### Direct Prediction

Decoder-only Transformer의 **causal self-attention**을 이용해
ETTh1의 `OT (Oil Temperature)`를 autoregressive 방식으로 예측합니다.

### 설정
```text
Target          : OT
Lookback        : 96
Prediction length: 24
Input           : Univariate
```

### 학습 목표
- Teacher forcing용 decoder sequence를 구성할 수 있습니다.
- `use_causal_mask=True`의 의미를 이해합니다.
- Decoder-only Transformer가 미래값을 autoregressive하게 생성하는 과정을 확인합니다.
- Direct prediction과 last-value baseline을 비교할 수 있습니다.

### Colab 데이터 경로
```text
MyDrive/Colab Notebooks/data/ETTh1.csv
```

In [ ]:
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from google.colab import drive

SEED = 42
keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("keras.ops:", hasattr(keras, "ops"))
print("GPU:", gpus)

drive.mount('/content/drive')

## 1. ETTh1 데이터 불러오기

In [ ]:
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/data/ETTh1.csv'

df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.head())
print(df.isna().sum())

ot = df[['OT']].values.astype('float32')


## 2. Chronological Split과 Standardization

시간 순서를 유지한 채 train / validation / test를 분리합니다.

Scaler는 training 구간에만 `fit`합니다.

In [ ]:
CONTEXT_LEN = 96
PRED_LEN = 24
MODEL_LEN = CONTEXT_LEN + PRED_LEN - 1   # 119

n = len(ot)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_raw = ot[:train_end]
val_raw = ot[train_end:val_end]
test_raw = ot[val_end:]

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_raw)
val_scaled = scaler.transform(val_raw)
test_scaled = scaler.transform(test_raw)

print('Train:', train_raw.shape)
print('Validation:', val_raw.shape)
print('Test:', test_raw.shape)

## 3. Teacher-Forcing Sequence 구성

학습에서는 실제 과거값과 ground-truth future의 이전 step을 decoder input으로 사용합니다.

예측 시에는 모델이 생성한 값을 다시 다음 step의 입력으로 사용합니다.

In [ ]:
def create_decoder_sequences(values, context_len=96, pred_len=24):
    total_len = context_len + pred_len
    X, Y = [], []

    for i in range(len(values) - total_len + 1):
        window = values[i:i + total_len]
        X.append(window[:-1])
        Y.append(window[context_len:])

    return (
        np.array(X, dtype='float32'),
        np.array(Y, dtype='float32')
    )

X_train, y_train = create_decoder_sequences(train_scaled, CONTEXT_LEN, PRED_LEN)
X_val, y_val = create_decoder_sequences(val_scaled, CONTEXT_LEN, PRED_LEN)
X_test, y_test = create_decoder_sequences(test_scaled, CONTEXT_LEN, PRED_LEN)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_val  :', X_val.shape)
print('y_val  :', y_val.shape)
print('X_test :', X_test.shape)
print('y_test :', y_test.shape)

## 4. Learned Positional Embedding

In [ ]:
class LearnedPositionalEmbedding(layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()
        self.position_embedding = layers.Embedding(
            input_dim=max_len,
            output_dim=embed_dim
        )

    def call(self, inputs):
        positions = keras.ops.arange(
            0, keras.ops.shape(inputs)[1], 1
        )
        position_embeddings = self.position_embedding(positions)
        return inputs + position_embeddings

## 5. Decoder-only Transformer Block

Self-attention에 `use_causal_mask=True`를 사용합니다.

따라서 각 position은 **자기 자신과 이전 position만** 볼 수 있고,
미래 정보는 attention에서 차단됩니다.

In [ ]:
class DecoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.dense1 = layers.Dense(ff_dim, activation='relu')
        self.dense2 = layers.Dense(embed_dim)

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=None):
        attention_output = self.attention(
            inputs,
            inputs,
            use_causal_mask=True,
            training=training
        )

        x = self.norm1(
            inputs + self.dropout1(attention_output, training=training)
        )

        ffn_output = self.dense2(self.dense1(x))

        return self.norm2(
            x + self.dropout2(ffn_output, training=training)
        )

## 6. Decoder-only Model 구성

In [ ]:
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

inputs = keras.Input(shape=(MODEL_LEN, 1))

projection_layer = layers.Dense(EMBED_DIM)
x = projection_layer(inputs)

position_layer = LearnedPositionalEmbedding(
    MODEL_LEN, EMBED_DIM
)
x = position_layer(x)

decoder1 = DecoderBlock(
    EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT
)
x = decoder1(x)

decoder2 = DecoderBlock(
    EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT
)
x = decoder2(x)

prediction_layer = layers.Dense(1)
all_predictions = prediction_layer(x)

backbone = keras.Model(
    inputs,
    all_predictions,
    name='decoder_only_backbone'
)

forecast_layer = layers.Lambda(
    lambda x: x[:, -PRED_LEN:, :]
)
forecast_outputs = forecast_layer(all_predictions)

model = keras.Model(
    inputs,
    forecast_outputs,
    name='decoder_only_forecaster'
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='mse',
    metrics=['mae']
)

model.summary()

## 7. Teacher Forcing으로 학습

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    shuffle=False,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Learning Curve')
plt.legend()
plt.grid(True)
plt.show()

## 8. Batched Autoregressive Inference

Inference에서는 ground-truth future를 사용할 수 없습니다.

```text
예측 1 → 다음 입력에 삽입
예측 2 → 다시 다음 입력에 삽입
...
```

이 과정을 24 step 반복합니다.

In [ ]:
@tf.function(reduce_retracing=True)
def autoregressive_forecast_batch(past_batch):
    past_batch = tf.cast(past_batch, tf.float32)
    batch_size = tf.shape(past_batch)[0]

    buffer = tf.concat(
        [
            past_batch,
            tf.zeros(
                (batch_size, PRED_LEN - 1, 1),
                dtype=tf.float32
            )
        ],
        axis=1
    )

    predictions = []

    for step in range(PRED_LEN):
        outputs = backbone(buffer, training=False)

        output_position = CONTEXT_LEN - 1 + step
        next_value = outputs[
            :,
            output_position:output_position + 1,
            :
        ]

        predictions.append(next_value)

        if step < PRED_LEN - 1:
            input_position = CONTEXT_LEN + step

            buffer = tf.concat(
                [
                    buffer[:, :input_position, :],
                    next_value,
                    buffer[:, input_position + 1:, :]
                ],
                axis=1
            )

    return tf.concat(predictions, axis=1)

## 9. Test Set 평가

In [ ]:
EVAL_BATCH_SIZE = 512

pred_batches = []
start_time = time.perf_counter()

for start in range(0, len(X_test), EVAL_BATCH_SIZE):
    end = min(start + EVAL_BATCH_SIZE, len(X_test))

    past_batch = tf.convert_to_tensor(
        X_test[start:end, :CONTEXT_LEN, :],
        dtype=tf.float32
    )

    pred_batch = autoregressive_forecast_batch(
        past_batch
    ).numpy()

    pred_batches.append(pred_batch[..., 0])

y_pred_ar = np.concatenate(pred_batches, axis=0)

elapsed = time.perf_counter() - start_time
print(f'Autoregressive evaluation time: {elapsed:.2f} s')

y_test_2d = y_test[..., 0]

norm_mse = mean_squared_error(
    y_test_2d.reshape(-1),
    y_pred_ar.reshape(-1)
)
norm_mae = mean_absolute_error(
    y_test_2d.reshape(-1),
    y_pred_ar.reshape(-1)
)

y_test_real = scaler.inverse_transform(
    y_test_2d.reshape(-1, 1)
).reshape(y_test_2d.shape)

y_pred_real = scaler.inverse_transform(
    y_pred_ar.reshape(-1, 1)
).reshape(y_pred_ar.shape)

real_mse = mean_squared_error(
    y_test_real.reshape(-1),
    y_pred_real.reshape(-1)
)
real_mae = mean_absolute_error(
    y_test_real.reshape(-1),
    y_pred_real.reshape(-1)
)

print(f'Normalized MSE : {norm_mse:.4f}')
print(f'Normalized MAE : {norm_mae:.4f}')
print(f'MSE (°C²)      : {real_mse:.4f}')
print(f'MAE (°C)       : {real_mae:.4f}')

## 10. Last-Value Baseline

마지막 관측값을 미래 24 step 동안 그대로 유지하는 단순 baseline입니다.

Transformer가 이 baseline보다 실제로 좋은지 반드시 확인합니다.

In [ ]:
last_value_pred = np.repeat(
    X_test[:, CONTEXT_LEN - 1, 0][:, None],
    PRED_LEN,
    axis=1
)

baseline_norm_mse = mean_squared_error(
    y_test_2d.reshape(-1),
    last_value_pred.reshape(-1)
)
baseline_norm_mae = mean_absolute_error(
    y_test_2d.reshape(-1),
    last_value_pred.reshape(-1)
)

last_value_real = scaler.inverse_transform(
    last_value_pred.reshape(-1, 1)
).reshape(last_value_pred.shape)

baseline_real_mse = mean_squared_error(
    y_test_real.reshape(-1),
    last_value_real.reshape(-1)
)
baseline_real_mae = mean_absolute_error(
    y_test_real.reshape(-1),
    last_value_real.reshape(-1)
)

print('Last-Value Baseline')
print(f'Normalized MSE : {baseline_norm_mse:.4f}')
print(f'Normalized MAE : {baseline_norm_mae:.4f}')
print(f'MSE (°C²)      : {baseline_real_mse:.4f}')
print(f'MAE (°C)       : {baseline_real_mae:.4f}')

## 11. Forecast Example

In [ ]:
sample_idx = 0

past_real = scaler.inverse_transform(
    X_test[sample_idx, :CONTEXT_LEN]
).reshape(-1)

future_real = y_test_real[sample_idx]
pred_real = y_pred_real[sample_idx]

past_x = np.arange(-CONTEXT_LEN + 1, 1)
future_x = np.arange(1, PRED_LEN + 1)

plt.figure(figsize=(10, 4))
plt.plot(past_x, past_real, label='Past OT')
plt.plot(future_x, future_real, label='Ground Truth')
plt.plot(future_x, pred_real, label='Decoder-only Forecast')

plt.axvline(0, linestyle='--')
plt.xlabel('Time Step')
plt.ylabel('OT (°C)')
plt.title('Autoregressive Forecast Example')
plt.legend()
plt.grid(True)
plt.show()

## 정리

### 핵심
- Decoder-only Transformer는 causal mask를 사용합니다.
- Training에서는 teacher forcing을 사용할 수 있습니다.
- Inference에서는 자신의 이전 prediction을 다시 입력하므로 error가 누적될 수 있습니다.
- Direct prediction은 미래값 자체를 예측합니다.

다음 notebook에서는 같은 구조에서 **residual prediction**으로 target 표현만 바꾸어 비교합니다.